In [34]:
import pandas as pd
import numpy as np
import pickle
import os
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

df1 = pd.read_csv("C:/Users/DELL XPS/OneDrive/Desktop/AIML/project_5/Data/Audible_Catlog.csv")
df2 = pd.read_csv("C:/Users/DELL XPS/OneDrive/Desktop/AIML/project_5/Data/Audible_Catlog_Advanced_Features.csv")

df = pd.merge(df1, df2, on=["Book Name", "Author"], how="left")

In [25]:
df.head()

,Book Name,Author,Rating_x,Number of Reviews_x,Price_x,Rating_y,Number of Reviews_y,Price_y,Description,Listening Time,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,313.0,10080.0,4.9,371.0,10080.0,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3658.0,615.0,4.6,3682.0,615.0,Brought to you by Penguin.,3 hours and 23 minutes,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20174.0,10378.0,4.4,20306.0,10378.0,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,4614.0,888.0,4.6,4678.0,888.0,Brought to you by Penguin.,5 hours and 35 minutes,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,4302.0,1005.0,4.6,4308.0,1005.0,"Stop going through life, Start growing throug...",6 hours and 25 minutes,",#6 in Audible Audiobooks & Originals (See Top..."


In [35]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle Rating
df['Rating'] = df[['Rating_x', 'Rating_y']].mean(axis=1)
df.drop(columns=['Rating_x', 'Rating_y'], inplace=True)

# Fill missing values
df['Rating'] = df['Rating'].fillna(df['Rating'].mean())
df['Description'] = df['Description'].fillna("No description")
df['Ranks and Genre'] = df['Ranks and Genre'].astype(str).str.lower()

# Clean text
df['Description'] = df['Description'].apply(lambda x: re.sub(r'[^a-zA-Z ]', '', x.lower()))

# NLP (TF-IDF)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Description'])

# Cosine Similarity
similarity = cosine_similarity(tfidf_matrix)

# Clustering
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(tfidf_matrix)
#create folder
os.makedirs("models",exist_ok=True)
# Save files
pickle.dump(similarity, open("models/similarity.pkl", "wb"))
pickle.dump(df, open("models/df.pkl", "wb"))

print("Model training completed!")

Model training completed!
